# 24 — The necessity test (E19): leave-EFG-out anchors (T2) and the conditional no-EFG ensemble (T3)

Runs on the curated-block re-solve (`VERSION = "v3"`, after 12 and 18). **T2** (~12 × ≤ 20 min): for every design
formulation, the anchor solved with every EFG multiplier at 0, certified to a 1e-3 gap (the EFG-free objective is a near-flat
plateau; 1e-4 took 25 min in v1 and > 100 min here — T2 is a witness test, so 1e-3 suffices; M4.28 addendum) (exactly as the leave-block-out anchors (E17 T3) did
for `efg_out`) → `runs_v3/e19_t2/<formulation_id>/run/portfolio.tif`. Core cells absent from every no-EFG anchor are
EFG-necessary by counterfactual (18c compares them with the adequacy-forced set (E19 T1)).
**T3** (~9 h + a guarded sweep): the full no-EFG ensemble (anchor + 50 MGA members + 50 guarded members per formulation)
— runs ONLY if `spec/v3/e19_gate.json`, written by 18c, says the core is predominantly forced (> 50%). Resumable;
live internet (WLS). Kernel `R (y2y)`.

In [1]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))
mpath <- pr_refresh_manifest(PROJ, ANALYSIS)
VERSION <- "v3.1"      # v3.1 = the curated block with window-derived targets (study plan v0.17.3)
stopifnot("the necessity test (E19) runs on the curated-block re-solve only (study plan v0.17.2)" = VERSION != "v1")
MANIFEST_REL <- sprintf("analyses/y2y/spec/manifest_%s.csv", VERSION); FREEZE_REL <- sprintf("analyses/y2y/spec/manifest_%s.sha256", VERSION)
RUNS_REL <- sprintf("analyses/y2y/runs_%s", VERSION); EFG_SUBDIR_EXPECTED <- paste0("iucn_efg_", sub("\\..*$", "", VERSION))
MAN <- read.csv(file.path(PROJ, MANIFEST_REL), stringsAsFactors = FALSE)
dig <- strsplit(readLines(file.path(PROJ, FREEZE_REL))[1], "  ")[[1]][1]
stopifnot(identical(unname(tools::sha256sum(file.path(PROJ, MANIFEST_REL))[[1]]), dig), nrow(MAN) == 12)
RUNS <- file.path(PROJ, RUNS_REL)
REAL245 <- "input_data/aligned_stack/climate_realizations/macrorefugia_245_2071_2100.tif"
ER <- jsonlite::read_json(file.path(PROJ, "analyses/y2y/spec/e_round_v13.json"))
BLOCKS <- lapply(ER$e17_t3$blocks, unlist); FLOOR_G <- 0.05
ctx585 <- pr_setup(mpath, PROJ); ctx585 <- modifyList(ctx585, pr_ingest(ctx585)); ctx585 <- modifyList(ctx585, pr_planning_units(ctx585))
ctx245 <- pr_setup(mpath, PROJ); ctx245$layers$path[ctx245$layers$name == "climate_type_macrorefugia"] <- REAL245
ctx245 <- modifyList(ctx245, pr_ingest(ctx245)); ctx245 <- modifyList(ctx245, pr_planning_units(ctx245))
efg_names <- ctx585$layers$name[ctx585$layers$role == "feature_efg"]
stopifnot(length(efg_names) > 0, all(grepl(paste0("/", EFG_SUBDIR_EXPECTED, "/"), ctx585$layers$path[ctx585$layers$role == "feature_efg"])))
base_for <- function(row) if (grepl("^ssp245", row$climate_level)) ctx245 else ctx585
form_wt  <- function(row) list(w = jsonlite::fromJSON(row$weight_vector), t = jsonlite::fromJSON(row$target_vector))
no_efg <- function(w) { for (f in efg_names) w[[f]] <- 0; w }        # every EFG multiplier -> 0 (E17 T3 convention)
# T2 solver settings (M4.28 addendum, 2026-09-14): the leave-EFG-out objective is a near-flat plateau (D ~ 1), and certifying
# 1e-4 on it took 25 min in v1 and > 100 min on v3.1's first formulation. T2 is a WITNESS test (a no-EFG plan that keeps a
# cell proves it is not EFG-necessary), so anchors are certified to 1e-3 -- fifty times inside the 5% band -- with a
# 20-minute cap; the achieved gap and bound are recorded per anchor and read back by 18c. T1 and the T3 gate are unaffected.
T2_GAP <- 1e-3; T2_TIME_LIMIT_S <- 1200
run_single <- function(base_ctx, w, t, out_rel, artifact = "run", opt_gap = T2_GAP, time_limit = T2_TIME_LIMIT_S) {
  done <- file.path(PROJ, out_rel, artifact, "run_summary.json")
  if (file.exists(done)) { cat(sprintf("   %s exists -- skipped\n", out_rel)); return(invisible(NULL)) }
  actx <- do.call(pr_override, c(list(base_ctx, targets = t, feature_weight_multipliers = w,
      results_dir = out_rel, results_subdir = artifact, solver = "gurobi", decision_type = "binary", opt_gap = opt_gap,
      solver_time_limit = time_limit, portfolio_n = 1)))
  actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing; actx$n_sol <- sv$n_sol; actx$sol_attrs <- sv$sol_attrs
  actx <- modifyList(actx, pr_summaries(actx)); pr_write_outputs(actx); invisible(NULL)
}
cat(sprintf("VERSION %s | %d design formulations | %d EFG features zeroed for the counterfactuals\n", VERSION, nrow(MAN), length(efg_names)))


manifest refreshed from config.py (analysis=y2y)
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y
ingested 28 features (8 continuous + 20 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 28 features to total=100000 each (scale-invariant conditioning)
planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
out

In [2]:
# ---- T2: leave-EFG-out anchors, one per design formulation (gap 1e-3, <= 20 min each) --------------------------------
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; wt <- form_wt(row)
  cat(sprintf("== %s (%d/%d)\n", row$formulation_id, i, nrow(MAN)))
  run_single(base_for(row), no_efg(wt$w), wt$t, file.path(RUNS_REL, "e19_t2", row$formulation_id))
  rs <- file.path(RUNS, "e19_t2", row$formulation_id, "run", "run_summary.json")
  if (file.exists(rs)) {
    pv <- jsonlite::read_json(rs)$solver_provenance
    cat(sprintf("   status %s | objective %.4f | bound %.4f | gap %.1e | %.0f s\n", pv$status, as.numeric(pv$objective), as.numeric(pv$objbound), as.numeric(pv$gap), as.numeric(pv$runtime)))
  }
}
cat("T2 complete -- next: 18c_e19_analysis (T1 + T2 agreement + the T3 gate)\n")


== s0_ssp585_theta5 (1/12)
   analyses/y2y/runs_v3.1/e19_t2/s0_ssp585_theta5 exists -- skipped
   status OPTIMAL | objective 4.8995 | bound 4.8990 | gap 1.0e-04 | 41 s
== s1_ssp585_theta5 (2/12)
   analyses/y2y/runs_v3.1/e19_t2/s1_ssp585_theta5 exists -- skipped
   status OPTIMAL | objective 4.6994 | bound 4.6985 | gap 2.0e-04 | 40 s
== s2_ssp585_theta5 (3/12)
   analyses/y2y/runs_v3.1/e19_t2/s2_ssp585_theta5 exists -- skipped
   status OPTIMAL | objective 5.0615 | bound 5.0615 | gap 2.4e-08 | 47 s
== s3_ssp585_theta5 (4/12)
   analyses/y2y/runs_v3.1/e19_t2/s3_ssp585_theta5 exists -- skipped
   status OPTIMAL | objective 5.0904 | bound 5.0904 | gap 3.0e-07 | 45 s
== s4_ssp585_theta3 (5/12)
   analyses/y2y/runs_v3.1/e19_t2/s4_ssp585_theta3 exists -- skipped
   status OPTIMAL | objective 4.5173 | bound 4.5172 | gap 4.6e-06 | 49 s
== s5_ssp585_theta5 (6/12)
   analyses/y2y/runs_v3.1/e19_t2/s5_ssp585_theta5 exists -- skipped
   status OPTIMAL | objective 11.1269 | bound 11.1264 | gap 0.0e+

In [3]:
# ---- E17 T3 on the curated block: leave-one-theme-out anchors at S0 (5 solves, ~5 min) --------------------------
# The E17 one-pager's bars were v1 evidence (runs/e17_t3, the 40-class block). On a v3.1 deck they must be v3.1:
# the four PROACT block-outs at the standard 1e-4 gap (15-67 s each in v1) and the EFG-out at the T2 witness gap.
row0 <- MAN[MAN$formulation_id == "s0_ssp585_theta5", ]; wt0 <- form_wt(row0)
for (b in names(BLOCKS)) {
  w <- wt0$w; for (f in unlist(BLOCKS[[b]])) w[[f]] <- 0
  cat(sprintf("== %s OUT\n", b))
  run_single(ctx585, w, wt0$t, file.path(RUNS_REL, "e17_t3", paste0(b, "_out")), opt_gap = 1e-4, time_limit = 43200)
}
cat("== efg OUT\n")
run_single(ctx585, no_efg(wt0$w), wt0$t, file.path(RUNS_REL, "e17_t3", "efg_out"))
cat("E17 T3 (curated block) complete -- re-run 19 then 20 for the one-pager\n")


== core_habitat OUT
  override targets          -> irrecoverable_carbon_m_soc=0.332, F1.1.web.mix_v2.0=0.5013, F2.1.web.alt_v4.0=0.4912, T6.1.web.map_v1.0=0.3588, F1.2.web.map_v1.0=0.2451, T2.2.web.mix_v1.0=0.1476, T3.4.web.mix_v1.0=0.1094, T5.4.web.mix_v1.0=0.1046, F1.6.web.mix_v1.0=0.1009, T4.4.web.orig_v1.0=0.1, SF1.2.web.orig_v1.0=0.1, T6.2.web.alt_v2.0=0.1, T6.3.web.map_v1.0=0.1, S1.1_SF1.1.web.merged_v3=0.1, T5.1.web.mix_v1.0=0.1, F1.3.web.map_v1.0=0.1, TF1.2.web.orig_v2.0=0.1, TF1.6_TF1.7.web.merged_v3=0.1, T6.4.web.orig_v1.0=0.1, F2.4.web.mix_v1.0=0.1, T2.1.web.mix_v1.0=0.1
  override feature_weight_multipliers -> climate_type_macrorefugia=0, transboundary_connectivity=0.669317, climate_corridors=1.17139, irrecoverable_carbon_m_soc=0.464638, irrecoverable_carbon_biomass=0.19856, aoh_richness_birds=1.32857, aoh_richness_mammals=1.70751
  override results_dir      -> analyses/y2y/runs_v3.1/e17_t3/core_habitat_out
  override results_subdir   -> run
  override solver           -> g

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0 and 1.707509)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 29 rows, 1272942 columns and 15032935 nonzeros (Min)
Model fingerprint: 0x38627d9b
Model has 27 linear objective coefficients
Variable types: 28 continuous, 1272914 integer (1272914 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range  [5e-02, 2e+00]
  Bounds range     [1e+00, 1e+0

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0 and 1.707509)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 29 rows, 1272942 columns and 15032935 nonzeros (Min)
Model fingerprint: 0xbb00f89d
Model has 26 linear objective coefficients
Variable types: 28 continuous, 1272914 integer (1272914 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range  [5e-02, 2e+00]
  Bounds range     [1e+00, 1e+0

...

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 29 rows, 1272942 columns and 15032935 nonzeros (Min)
Model fingerprint: 0x7272450e
Model has 26 linear objective coefficients
Variable types: 28 continuous, 1272914 integer (1272914 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range  [5e-02, 2e+00]
  Bounds range     [1e+00, 1e+0

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0 and 1.460018)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 29 rows, 1272942 columns and 15032935 nonzeros (Min)
Model fingerprint: 0x2205c9bc
Model has 26 linear objective coefficients
Variable types: 28 continuous, 1272914 integer (1272914 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range  [5e-02, 1e+00]
  Bounds range     [1e+00, 1e+0

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0 and 1.707509)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.001, `time_limit` = 1200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 1200
Set parameter MIPGap to value 0.001
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  1200
MIPGap  0.001
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 29 rows, 1272942 columns and 15032935 nonzeros (Min)
Model fingerprint: 0x5cf7567a
Model has 8 linear objective coefficients
Variable types: 28 continuous, 1272914 integer (1272914 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range 

In [4]:
# ---- T3 (CONDITIONAL): the no-EFG ensemble -- anchors + MGA + guarded members, EFG multipliers 0 --------------
gate_f <- file.path(PROJ, sprintf("analyses/y2y/spec/%s/e19_gate.json", VERSION))
gate <- if (file.exists(gate_f)) jsonlite::read_json(gate_f) else NULL
if (is.null(gate)) {
  cat("T3 gate not written yet -- run 18c_e19_analysis first (it decides whether the core is predominantly forced)\n")
} else if (!isTRUE(gate$t3_triggered)) {
  cat(sprintf("T3 NOT triggered: forced share of the core %.1f%% (rule: > 50%%) -- the no-EFG ensemble is not run\n", 100 * gate$forced_share_core_all))
} else {
  cat(sprintf("T3 TRIGGERED: forced share of the core %.1f%% -- solving the no-EFG ensemble (~9 h + guarded)\n", 100 * gate$forced_share_core_all))
  for (i in seq_len(nrow(MAN))) {
    row <- MAN[i, ]; wt <- form_wt(row); cd <- file.path(RUNS, "e19_t3", row$formulation_id); dir.create(cd, recursive = TRUE, showWarnings = FALSE)
    cat(sprintf("\n===================== %s (%d/%d) =====================\n", row$formulation_id, i, nrow(MAN)))
    if (file.exists(file.path(cd, "mga_guard_g05.tif"))) { cat("   exists -- skipped\n"); next }
    actx <- pr_override(base_for(row), targets = wt$t, feature_weight_multipliers = no_efg(wt$w),
        results_dir = file.path(RUNS_REL, "e19_t3", row$formulation_id), results_subdir = "mga_build",
        solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)
    actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
    bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
    cm <- mga_compile(actx); anchor <- mga_anchor(cm, opt_gap = row$opt_gap)
    if (!file.exists(file.path(cd, "mga_g05.tif"))) {
      gen <- mga_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested); mga_write(gen, cm, actx$cost, cd, "g05")
    }
    r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r)); v[cm$pu_index] <- as.integer(anchor$x); terra::values(r) <- v
    terra::writeRaster(r, file.path(cd, "anchor.tif"), overwrite = TRUE, datatype = "INT1U", NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
    jsonlite::write_json(list(formulation_id = row$formulation_id, experiment = "E19 T3 no-EFG ensemble", anchor_objective = anchor$z,
                              anchor_gap = anchor$gap, anchor_runtime_s = anchor$runtime, weight_vector = no_efg(wt$w), target_vector = wt$t,
                              k = row$k_requested, g = row$band_gap_g, created_utc = format(Sys.time(), tz = "UTC")),
                         file.path(cd, "formulation_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
    gg <- mga_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested, floors = list(ctx = actx, blocks = BLOCKS, g = FLOOR_G))
    layers <- lapply(seq_len(gg$k), function(j) { rr <- terra::rast(actx$cost); vv <- rep(NA_integer_, terra::ncell(rr)); vv[cm$pu_index] <- as.integer(gg$members[j, ]); terra::values(rr) <- vv; rr })
    s <- terra::rast(layers); names(s) <- sprintf("guard_%02d", seq_len(gg$k))
    terra::writeRaster(s, file.path(cd, "mga_guard_g05.tif"), overwrite = TRUE, datatype = "INT1U", NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
    write.csv(gg$certificates, file.path(cd, "certificates_guard.csv"), row.names = FALSE)
    cat(sprintf("   wrote anchor, MGA members, guarded members for %s\n", row$formulation_id))
  }
  cat("T3 complete -- re-run 18c_e19_analysis for the F_noEFG surface\n")
}


T3 NOT triggered: forced share of the core 0.0% (rule: > 50%) -- the no-EFG ensemble is not run
